<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/sce_RAG_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시나리오 RAG (잠긴 종) - 정리본

시나리오 카드(`cards.json`)를 검색해서, 지금 플레이어가 볼 수 있는 카드만 가져오는 RAG

| 목차 | 하는 일 | 셀 |
|---|---|---|
| L1 | 카드 읽기, 공개/GM 전용 나누기 | ##1~##2 |
| L2 | 현재 상태에서 검색 가능한 카드만 거르는 규칙 | ##3 |
| L3 | 카드 → 검색용 청크 | ##4 |
| L4 | 임베딩 + Chroma 저장 | ##5 |
| L5 | 상태 필터 검색 | ##6 |
| L6 | 골든셋 평가 (Hit@1, Hit@3, MRR) | ##7~##8 |
| L7 | Qwen으로 장면 서술 | ##9~##10 |

In [1]:
# ===== ##0 설치와 import 한 곳에 모으기 (런타임을 바꾸면 이 셀부터 실행) =====
!pip install -q chromadb accelerate

import json
import re
import requests
from collections import Counter

import chromadb
from sentence_transformers import SentenceTransformer

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
t

# L1. 시나리오 카드 정리 (##1~##2)

카드 파일을 읽고, 검색 대상(플레이어에게 보여 줄 수 있는 카드)과 GM 전용 카드를 나눕니다.
비밀이 검색에 섞이지 않게 하는 것이 목적

In [2]:
## 1. cards.json 가져오기 + 구조 확인

CARDS_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/Kim/TheDrownedBell/game/data/cards.json"
response = requests.get(CARDS_URL, timeout=60)
response.raise_for_status()
with open("cards.json", "wb") as f:
    f.write(response.content)

with open("cards.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cards = data["cards"]      # 카드 166장
rooms = data["rooms"]      # 방 id -> 한글 이름표 (27개)

print("카드 개수 :", len(cards), "| 방 개수 :", len(rooms))
print("카드 하나의 키 :", list(cards[0].keys()))
for key in ["kind", "owner", "scene_id"]:
    print(key, dict(Counter(card[key] for card in cards)))

# <출력>
# 카드 개수 : 166 | 방 개수 : 27
# kind {'desc': 33, 'npc': 10, 'clue': 55, 'secret': 15, 'check': 30, 'monster': 9, 'ending': 9, 'rumor': 5}
# owner {'all': 133, 'cleric': 11, 'rogue': 8, 'wizard': 9, 'fighter': 5}
# scene_id {1: 34, 2: 16, 3: 37, 4: 28, 5: 26, 6: 25}

카드 개수 : 166 | 방 개수 : 27
카드 하나의 키 : ['id', 'scene_id', 'reveal_after', 'owner', 'kind', 'req_flag', 'text', 'room', 'label', 'player_text', 'gm_only_text', 'gm_note', 'source']
kind {'desc': 33, 'npc': 10, 'clue': 55, 'secret': 15, 'check': 30, 'monster': 9, 'ending': 9, 'rumor': 5}
owner {'all': 133, 'cleric': 11, 'rogue': 8, 'wizard': 9, 'fighter': 5}
scene_id {1: 34, 2: 16, 3: 37, 4: 28, 5: 26, 6: 25}


In [3]:
## 2. 공개용 / GM 전용 나누기 + 검증
# player_text가 있는 카드만 검색 대상으로 삼는다 (GM 전용 카드는 player_text가 비어 있음)

search_cards = [c for c in cards if c["player_text"]]
print("검색 대상 카드 :", len(search_cards), "| GM 전용 카드 :", len(cards) - len(search_cards))

# 비밀 종류 카드가 섞이지 않았는지 검증
kinds = Counter(c["kind"] for c in search_cards)
for banned in ["secret", "check", "monster"]:
    assert banned not in kinds, banned
print("secret / check / monster 카드가 검색 대상에 없음을 확인")

# 공개 카드인데 GM 문장이 일부 섞인 카드 (이 문장은 청크에 넣지 않는다)
print("GM 문장이 섞인 공개 카드 :", [c["id"] for c in search_cards if c["gm_only_text"]])

# <출력>
# 검색 대상 카드 : 112 | GM 전용 카드 : 54
# GM 문장이 섞인 공개 카드 9장

검색 대상 카드 : 112 | GM 전용 카드 : 54
secret / check / monster 카드가 검색 대상에 없음을 확인
GM 문장이 섞인 공개 카드 : ['scn-s1-npc-halvik', 'scn-s1-clue-well', 'scn-s2-clue-newstone', 'scn-s3-clue-bodies', 'scn-s3-clue-door', 'scn-q4-payoff-lullaby', 'scn-h-fighter-periapt', 'scn-h-rest-infirmary', 'scn-m-npc-innkeeper']


# L2. 검색 규칙 정하기 (##3)

"지금 이 플레이어에게 보여도 되는 카드인가?"를 판단하는 함수입니다.
의미가 비슷해도 아직 공개 전이거나, 가 본 적 없는 방이거나, 다른 직업 전용이면 검색에서 뺍니다.

In [4]:
## 3. 현재 상황에서 검색 가능한 카드만 거르기

def can_search(card, state):
    if state["scene"] < card["reveal_after"]:
        return False        # 아직 공개 장면이 아님
    if card["req_flag"] and card["req_flag"] not in state["flags"]:
        return False        # 필요한 플래그가 없음 (req_flag가 빈 글자면 이 검사는 건너뜀)
    if card["room"] and card["room"] not in state["visited"]:
        return False        # 아직 가 보지 않은 방
    if card["owner"] != "all" and card["owner"] != state["cls"]:
        return False        # 다른 직업 전용 카드
    return True

# 예시 상황: 장면 1, 전사, 마을 광장에만 와 본 상태
state = {"scene": 1, "flags": set(), "visited": {"village_square"}, "cls": "fighter"}
found = [c for c in search_cards if can_search(c, state)]
print("검색 가능한 카드 :", len(found))
print([c["id"] for c in found])

검색 가능한 카드 : 5
['scn-s1-desc', 'scn-s1-clue-missing', 'scn-q1-npc-sera', 'scn-g-rumors', 'scn-v8-rumor-bell']


# L3. 시나리오 청크 만들기 (##4)

카드 1장 = 청크 1개. 청크는 (1) 검색에 쓸 글, (2) 필터용 메타데이터, (3) 고유 id로 구성됩니다.
GM 전용 문장은 청크에 넣지 않습니다. (청크는 나중에 Qwen에게 넘어가므로, 들어가면 비밀이 샙니다.)

In [5]:
## 4. 청크 만들기 + 검증

card_by_id = {c["id"]: c for c in cards}      # id로 카드 원본을 바로 찾기 위한 사전

def make_chunk_text(card):
    # "[종류] 방이름" 머리말 + player_text (GM 문장 제외)
    room_name = rooms.get(card["room"], "")   # 엔딩 카드 9장은 room이 비어 있어서 .get 사용
    header = f"[{card['label']}]"
    if room_name:
        header += f" {room_name}"
    return header + "\n" + card["player_text"]

def make_metadata(card):
    # 비밀이나 아직 안 열린 카드를 걸러내는 데 쓰는 꼬리표
    return {
        "card_id": card["id"],
        "scene_id": card["scene_id"],
        "reveal_after": card["reveal_after"],
        "owner": card["owner"],
        "kind": card["kind"],
        "req_flag": card["req_flag"],
        "room": card["room"],
    }

chunks = [{"id": c["id"], "text": make_chunk_text(c), "metadata": make_metadata(c)}
          for c in search_cards]

# 검증: 개수 / GM 문장 없음 / id 중복 없음
print("청크 개수 :", len(chunks))
for c, ch in zip(search_cards, chunks):
    if c["gm_only_text"]:
        assert c["gm_only_text"] not in ch["text"], c["id"]
print("GM 문장이 청크에 없음을 확인")
print("id 중복 :", len(chunks) - len({ch["id"] for ch in chunks}))

lengths = sorted(len(ch["text"]) for ch in chunks)
print("청크 길이 min / 중앙 / max :", lengths[0], lengths[len(lengths) // 2], lengths[-1])
print()
print(chunks[0]["text"])

청크 개수 : 112
GM 문장이 청크에 없음을 확인
id 중복 : 0
청크 길이 min / 중앙 / max : 92 153 304

[장면 묘사] 우물 광장
에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.


# L4. 임베딩과 Chroma 저장 (##5)

청크 글을 벡터(숫자 384개)로 바꿔서 Chroma에 저장합니다. 글, 이름표(metadata), 벡터를 같이 저장합니다.
임베딩 모델: `intfloat/multilingual-e5-small` (글 앞에 `passage: `, 질문 앞에 `query: `를 붙입니다)

In [6]:
## 5. 임베딩 + Chroma 저장

model = SentenceTransformer("intfloat/multilingual-e5-small")

texts = ["passage: " + ch["text"] for ch in chunks]
vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print("벡터 모양 :", vectors.shape)

client = chromadb.Client()      # 메모리에만 저장 (런타임을 끄면 사라짐)
collection = client.get_or_create_collection(
    name="scenario_cards",
    metadata={"hnsw:space": "cosine"},
)
collection.upsert(
    ids=[ch["id"] for ch in chunks],
    documents=[ch["text"] for ch in chunks],
    metadatas=[ch["metadata"] for ch in chunks],
    embeddings=vectors.tolist(),
)
print("저장된 개수 :", collection.count())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

벡터 모양 : (112, 384)
저장된 개수 : 112


# L5. 시나리오 검색 + 상태 필터 (##6)

Chroma에서 유사도 순으로 전부 가져온 뒤, `can_search`를 통과한 카드만 남기고 앞에서 k개를 돌려줍니다.

In [7]:
## 6. 상태를 반영한 검색 함수

def search(question, state, k=3, use_filter=True):
    q_vec = model.encode("query: " + question, normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_vec.tolist()], n_results=len(chunks))

    hits = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        if use_filter and not can_search(card_by_id[cid], state):
            continue                          # 지금 볼 수 없는 카드는 건너뜀
        hits.append((cid, round(1 - dist, 3)))    # (카드 id, 유사도)
    return hits[:k]

# 간단 시험
st = {"scene": 1, "flags": set(), "visited": {"village_square"}, "cls": "fighter"}
for q in ["우물에 돌을 떨어뜨려 본다", "마을 사람에게 말을 건다"]:
    print(q, "->", search(q, st))

우물에 돌을 떨어뜨려 본다 -> [('scn-s1-desc', 0.865), ('scn-s1-clue-missing', 0.851), ('scn-q1-npc-sera', 0.837)]
마을 사람에게 말을 건다 -> [('scn-v8-rumor-bell', 0.855), ('scn-s1-desc', 0.854), ('scn-g-rumors', 0.853)]


# L6. 골든셋과 검색 평가 (##7~##8)

"이 질문에는 이 카드가 정답"이라고 적어 둔 시험 문제집으로 검색을 숫자로 잽니다.
- `golden` : 개발용 14문제 (실패 원인을 보고 고치는 데 써도 됨)
- `golden_mine` : 직접 만든 평가용 4문제 (**이 결과를 보고 코드를 고치지 않기**)

유사도와 순위는 참고용입니다. 최종 판단은 가져온 카드를 직접 읽고 합니다.

In [8]:
## 7. 골든셋 만들기

def make_state(scene, cls):
    # 그 장면까지 열린 방을 모두 가 본 상태로 만든다
    visited = {c["room"] for c in cards if c["reveal_after"] <= scene and c["room"]}
    return {"scene": scene, "flags": set(), "visited": visited, "cls": cls}

golden = [
    {"q": "우물을 덮은 돌판을 자세히 살핀다",          "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-well"]},
    {"q": "촌장에게 마을에서 있었던 일을 물어본다",    "scene": 1, "cls": "fighter", "answers": ["scn-s1-npc-halvik"]},
    {"q": "눈이 안 보이는 할머니에게 말을 걸어 본다",  "scene": 1, "cls": "fighter", "answers": ["scn-s1-npc-mera"]},
    {"q": "사라진 사람들에 대해 알아본다",             "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-missing"]},
    {"q": "마을 사람들이 무슨 소문을 말하는지 듣는다", "scene": 1, "cls": "fighter", "answers": ["scn-g-rumors", "scn-v8-rumor-bell"]},
    {"q": "대장장이에게 무기를 사고 싶다",             "scene": 1, "cls": "fighter", "answers": ["scn-m-npc-smith", "scn-m-shop-goods"]},
    {"q": "여관에서 쉬어 가고 싶다",                   "scene": 1, "cls": "fighter", "answers": ["scn-m-npc-innkeeper"]},
    {"q": "촌장 집 벽난로 위에 걸린 물건을 본다",      "scene": 1, "cls": "cleric",  "answers": ["scn-s1-clue-crest", "scn-g-halvik-house"]},
    {"q": "촌장의 손을 유심히 살핀다",                 "scene": 1, "cls": "rogue",   "answers": ["scn-s1-clue-hands"]},
    {"q": "밤에 우물가에서 들리는 소리에 귀를 기울인다", "scene": 1, "cls": "fighter", "answers": ["scn-g-well-night"]},
    {"q": "우물 벽을 타고 내려간다",                   "scene": 2, "cls": "fighter", "answers": ["scn-s2-desc", "scn-g-shaft-stone"]},
    {"q": "중간 턱의 옆 통로를 조사한다",              "scene": 2, "cls": "fighter", "answers": ["scn-s2-clue-newstone"]},
    {"q": "공기가 어디서 흘러오는지 느껴 본다",        "scene": 2, "cls": "wizard",  "answers": ["scn-s2-clue-airflow"]},
    {"q": "갱도 바닥에 숨겨진 것이 있는지 찾아본다",   "scene": 2, "cls": "fighter", "answers": ["scn-h-fighter-periapt"]},
]

# 평가용 (이 문제를 보고 코드를 고치지 않기)
golden_mine = [
    {"q": "최근에 실종한 사람이 있냐고 물어본다",     "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-missing"]},
    {"q": "할빅의 손을 유심히 본다",                 "scene": 1, "cls": "rogue",   "answers": ["scn-s1-clue-hands"]},
    {"q": "인형을 안 받고 찾아주겠다고 약속한다",     "scene": 1, "cls": "fighter", "answers": ["scn-q1-npc-sera"]},
    {"q": "할빅에게 지금까지 있었던일을 물어본다",   "scene": 2, "cls": "fighter", "answers": ["scn-s1-npc-halvik"]},
]

# 정답 카드가 그 상황에서 검색 가능한지 확인
for g in golden + golden_mine:
    st = make_state(g["scene"], g["cls"])
    for a in g["answers"]:
        assert can_search(card_by_id[a], st), (g["q"], a)
print("골든셋 :", len(golden), "+", len(golden_mine), "문제 / 정답 카드가 모두 그 상황에서 검색 가능함을 확인")

골든셋 : 14 + 4 문제 / 정답 카드가 모두 그 상황에서 검색 가능함을 확인


In [9]:
## 8. 평가 함수 + 필터 없음/있음 비교

def evaluate(items, use_filter=True):
    ranks = []
    for g in items:
        st = make_state(g["scene"], g["cls"])
        hits = [cid for cid, _ in search(g["q"], st, k=10, use_filter=use_filter)]
        rank = None
        for i, cid in enumerate(hits):
            if cid in g["answers"]:
                rank = i + 1              # 정답이 처음 나온 등수
                break
        ranks.append(rank)

    n = len(ranks)
    hit1 = sum(r == 1 for r in ranks) / n                          # 1등으로 찾은 비율
    hit3 = sum(r is not None and r <= 3 for r in ranks) / n        # 3등 안에 찾은 비율
    mrr = sum(1 / r for r in ranks if r) / n                       # 등수 역수의 평균
    return ranks, hit1, hit3, mrr

for name, items in [("개발용 14문제", golden), ("직접 만든 4문제", golden_mine)]:
    for use_filter in (False, True):
        ranks, hit1, hit3, mrr = evaluate(items, use_filter)
        print(f"{name} | 필터 {'있음' if use_filter else '없음'} | Hit@1 {hit1:.2f} | Hit@3 {hit3:.2f} | MRR {mrr:.2f}")

print()
ranks = evaluate(golden, True)[0]
for g, r in zip(golden, ranks):
    print(r, "|", g["q"])

# <이전 실행 결과>
# 개발용 14 : 필터 없음 0.43 / 0.79 / 0.61  ->  필터 있음 0.64 / 0.86 / 0.76
# 직접 만든 4 : 필터 없음 Hit@3 0.75, MRR 0.80  ->  필터 있음 Hit@3 1.00, MRR 0.88

개발용 14문제 | 필터 없음 | Hit@1 0.43 | Hit@3 0.79 | MRR 0.61
개발용 14문제 | 필터 있음 | Hit@1 0.64 | Hit@3 0.86 | MRR 0.76
직접 만든 4문제 | 필터 없음 | Hit@1 0.75 | Hit@3 0.75 | MRR 0.80
직접 만든 4문제 | 필터 있음 | Hit@1 0.75 | Hit@3 1.00 | MRR 0.88

1 | 우물을 덮은 돌판을 자세히 살핀다
2 | 촌장에게 마을에서 있었던 일을 물어본다
2 | 눈이 안 보이는 할머니에게 말을 걸어 본다
7 | 사라진 사람들에 대해 알아본다
1 | 마을 사람들이 무슨 소문을 말하는지 듣는다
1 | 대장장이에게 무기를 사고 싶다
1 | 여관에서 쉬어 가고 싶다
1 | 촌장 집 벽난로 위에 걸린 물건을 본다
1 | 촌장의 손을 유심히 살핀다
1 | 밤에 우물가에서 들리는 소리에 귀를 기울인다
None | 우물 벽을 타고 내려간다
2 | 중간 턱의 옆 통로를 조사한다
1 | 공기가 어디서 흘러오는지 느껴 본다
1 | 갱도 바닥에 숨겨진 것이 있는지 찾아본다


# L6 결과 메모: 골든셋에서 발견한 것과 문제점

## 결과 요약
| 실험 | 비교 | 결과 | 비고 |
|---|---|---|---|
| 필터 효과 (개발용 14문제) | 필터 없음 → 있음 | Hit@1 0.43 → 0.64, Hit@3 0.79 → 0.86, MRR 0.61 → 0.76 | 정답 1등 6개 → 9개 |
| 필터 효과 (직접 만든 4문제) | 필터 없음 → 있음 | Hit@3 0.75 → 1.00, MRR 0.80 → 0.88 | 표본이 작아 방향만 참고 |
| 머리말 효과 (14문제) | 머리말 있음 → 없음 | Hit@1 0.64 → 0.57, MRR 0.76 → 0.74 | 좋아진 문제 2, 나빠진 문제 2, 같은 문제 10 → 뚜렷한 효과 없음 (실험 코드는 정리하며 삭제, 결과만 남김) |

## 발견한 문제점
| 번호 | 문제 | 원인 (확인됨/추정) | 개선 방향 |
|---|---|---|---|
| 1 | 필터 없이 검색하면 지금 볼 수 없는 카드가 상위에 섞임 (15자리 중 12개) | 의미 검색은 글만 비교하고 게임 상태를 모름 (확인) | can_search 필터 적용 (완료) |
| 2 | 코사인 점수가 0.80 ~ 0.89에 몰려 1~5등 차이가 0.01 이내 | e5-small의 점수 분포 특성으로 추정 (미확인) | bge-m3와 비교 예정 |
| 3 | "사라진 사람들에 대해 알아본다"의 정답이 7등 | 사람이 나오는 NPC, 소문 카드가 더 높게 나옴 (원인 미확인) | 질문 추가 후 재확인 |
| 4 | "우물 벽을 타고 내려간다"의 정답이 10등 밖 | 머리말을 빼면 2등으로 올라와 머리말 영향 가능성 (추정) | 머리말 형식 재검토 |
| 5 | 만능 카드(scn-s1-desc)가 여러 질문에서 1등 | 원인 미확인 | 질문을 늘려 패턴 확인 |
| 6 | 시나리오에 없는 행동에도 카드가 나옴 | 검색은 항상 가장 비슷한 것을 돌려주고 "없음"을 구분하는 점수가 없음 (확인) | 점수 기준선 대신 엔진과 의도 분류가 판단 |
| 7 | 직접 만든 질문 8개 중 4개가 카드로 답할 수 없는 행동 | 카드 검색이 아니라 행동 해석이나 룰북 규칙 문제 (추정) | 의도 분류와 룰북 RAG에서 처리 |

## 한계
- 문제 수가 적다: 개발용 14개, 평가용 4개. 한 문제가 7~25%포인트라 작은 차이는 우연일 수 있다.
- 필터 후 후보가 19~27장뿐이라 우연히 맞기 쉽다.
- 개발용 14문제는 실패 원인을 본 뒤라 평가용으로 쓸 수 없다.

## 남은 일
- 직접 쓴 평가용 질문을 8개 이상으로 늘리기
- bge-m3와 e5-small을 같은 골든셋으로 비교
- 머리말 형식 재검토

# L7. 장면 서술 (##9~##10)

시나리오 검색 결과(카드의 `player_text`)만 Qwen에게 주고 장면을 서술하게 합니다.
`player_text`만 쓰므로 GM 전용 문장은 모델에 들어가지 않습니다.
평가는 점수가 아니라 서술을 직접 읽고 ○/△/×로 합니다: 필요한 정보 포함 / 카드에 없는 내용을 지어냄 / 비밀 누설.

In [10]:
## 9. Qwen 불러오기 + 질문 보내고 답 받기

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꾸세요)")

MODEL_NAME = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")

def ask_llm(messages, max_new_tokens=300):
    # messages -> 모델이 읽는 글로 바꾸기 (enable_thinking=False: 생각 과정 출력 끄기)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]     # 질문 부분은 빼고 새로 만든 부분만
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(ask_llm([{"role": "user", "content": "안녕, 한 문장으로 자기소개 해줘."}]))

GPU : Tesla T4


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

안녕하세요, 저는 AI 어시스턴트입니다. 도움이 필요하시면 언제든지 말씀해주세요.


In [11]:
## 10. 장면 서술 (시나리오 검색 -> 카드 문장만으로 Qwen 서술)
# 규칙: 카드에 없는 사실 금지, 다음 행동 지시 금지 (게임의 narrator.py와 같은 방향)

def find_english(text):
    # 서술에 영어 단어가 섞였는지 확인 (3글자 이상)
    return re.findall(r"[A-Za-z][A-Za-z'-]{2,}", text)

def narrate_scene(text, state, k=2):
    hits = search(text, state, k=k)                                   # [(카드 id, 유사도), ...] (상태 필터 적용)
    shown = "\n".join(f"- {card_by_id[cid]['player_text']}" for cid, sim in hits)
    messages = [
        {"role": "system", "content": "너는 한국어 TRPG의 진행자(마스터)다. 아래 [이번에 드러난 정보]에 적힌 내용만으로 장면을 2~4문장, 현재형으로 묘사한다. 적히지 않은 사실을 지어내거나 암시하지 않는다. 플레이어가 다음에 무엇을 해야 하는지 지시하지 않는다."},
        {"role": "user", "content": f"[플레이어 입력]\n{text}\n\n[이번에 드러난 정보]\n{shown}\n\n위 정보만 바탕으로 장면을 묘사해 줘."},
    ]
    return hits, ask_llm(messages, max_new_tokens=300)

for g in golden_mine:
    st = make_state(g["scene"], g["cls"])
    hits, ans = narrate_scene(g["q"], st)
    print("입력:", g["q"])
    print("가져온 카드:", [cid for cid, sim in hits], "| 정답:", g["answers"])
    print("서술:", ans)
    print("남은 영어:", find_english(ans))
    print("-" * 40)

입력: 최근에 실종한 사람이 있냐고 물어본다
가져온 카드: ['scn-s1-clue-missing', 'scn-g-rumors'] | 정답: ['scn-s1-clue-missing']
서술: 우물 근처는 어두운 밤에 더욱 조용하고 차가워 보인다. 마을 사람들은 늑대나 도적을 의심하며 서로를 바라보는 것이 많다. 대장간 노인의 말에 따르면, 종소리가 매일 더 가까워지고 있다. 여관 주인은 밤에는 우물 곁을 지나지 말라고 경고한다. 마을 사람들은 실종자 가족을 만나려면 세라를 찾아가야 한다고 말한다.
남은 영어: []
----------------------------------------
입력: 할빅의 손을 유심히 본다
가져온 카드: ['scn-s1-clue-hands', 'scn-s1-npc-halvik'] | 정답: ['scn-s1-clue-hands']
서술: 할빅은 큰 어깨를 펴고 일행을 맞이한다. 그의 손은 채석공으로서의 경험을 간직했지만, 최근의 흔적들이 눈에 띄는 편이다. 손바닥 아래에 있는 물집과 회색 진흙은 그가 최근에 지나간 곳을 말해준다. 그는 우물에 대해 말하면서도 시선을 그곳으로 돌리지 않는다.
남은 영어: []
----------------------------------------
입력: 인형을 안 받고 찾아주겠다고 약속한다
가져온 카드: ['scn-q4-npc-mera-song', 'scn-q1-npc-sera'] | 정답: ['scn-q1-npc-sera']
서술: 우물가에서 앉은 메라는 손을 흔들며 노래를 부르고 있다. 그녀의 뒤에 세라가 앉아, 낮은 목소리로 속삭인다. "찾을 수 없으면 그렇다고만 말해 줘요." 세라의 손가락은 어릴 적 딸이 들고 다녔던 인형을 가리키며, "이걸 보면 그 애가 알아볼 거예요"라고 말한다. 우물 주변은 조용하고, 어릴 적 할머니가 불러曾던 노래의 흐름이 여전히 공중을 지나간다.
남은 영어: []
----------------------------------------
입력: 할빅에게 지금까지 있었던

In [ ]:
## 11. 플레이어처럼 자유 행동 입력해 보기
# 장면/직업을 바꾸려면 아래 두 줄만 고치세요. 장면 1은 마을, 장면 2는 우물 아래입니다.
SCENE = 1
CLS = "fighter"      # fighter / cleric / rogue / wizard

st = make_state(SCENE, CLS)
while True:
    text = input("\n[자유 행동] > ").strip()
    if not text:
        break
    hits, ans = narrate_scene(text, st)
    print("가져온 카드:")
    for cid, sim in hits:
        print(f"  - {cid} ({sim}) {card_by_id[cid]['player_text'][:50]}...")
    print("\nGM:", ans)
    print("남은 영어:", find_english(ans))